# Document Splitting

In [1]:
pip install langchain

Note: you may need to restart the kernel to use updated packages.


In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

## Splitter mechanics

In [3]:
chunk_size = 26
chunk_overlap = 4

r_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
c_splitter = CharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

In [4]:
text1 = 'abcdefghijklmnopqrstuvwxyz'
r_splitter.split_text(text1)

['abcdefghijklmnopqrstuvwxyz']

In [5]:
text2 = 'abcdefghijklmnopqrstuvwxyzabcdefg'
r_splitter.split_text(text2)

['abcdefghijklmnopqrstuvwxyz', 'wxyzabcdefg']

### Arabic example

In [6]:
arabic_text = "المادة الأولى يعد باطلا كل شرط يخالف احكام هذا القانون ما لم يكن الشرط اكثر فائدة للعامل"
r_splitter.split_text(arabic_text)

['المادة الأولى يعد باطلا كل',
 'كل شرط يخالف احكام هذا',
 'هذا القانون ما لم يكن',
 'يكن الشرط اكثر فائدة',
 'للعامل']

## Real project data — one chunk per article / per case

In [ ]:
import json
import re
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

PROCESSED_DIR = Path("../data/processed")

# sjc.bh case type codes (see 01_scraping/01a)
SJC_TYPE_CODES = {"M": "مدني", "J": "جنائي", "S": "شرعي", "T": "تجاري", "E": "انتخابات", "P": "توحيد المبادئ"}

ORDINAL_WORDS = ["الاولي", "الأولى", "الثانية", "الثالثة", "الرابعة", "الخامسة",
                 "السادسة", "السابعة", "الثامنة", "التاسعة", "العاشرة"]
ORDINAL_TO_DIGIT = {w: str(i + 1) for i, w in enumerate(
    ["الاولي", "الثانية", "الثالثة", "الرابعة", "الخامسة",
     "السادسة", "السابعة", "الثامنة", "التاسعة", "العاشرة"]
)}
ORDINAL_TO_DIGIT["الأولى"] = "1"

# Compound numbers (e.g. treaty articles like "1.1", "2.3") captured fully, not truncated at the
# first digit.
ARTICLE_HEADER = re.compile(
    r"(?:المادة|مادة)\s*(?:\(\s*(\d+(?:\.\d+)*)\s*\)|(\d+(?:\.\d+)*)|" + "|".join(ORDINAL_WORDS) + r")"
)

# Excludes citation-style article references (e.g. "...pursuant to Article 66...") from being
# treated as boundaries — CITATION_PHRASE detects "of this law/constitution/etc." wording nearby.
CITATION_PHRASE = re.compile(
    r"من\s*(?:هذا\s+|هذه\s+)?(?:القانون|الدستور|المرسوم|النظام|الميثاق|القرار|اللائحة|الاتفاقية|البروتوكول|بروتوكول)\b|منه\b"
)

# BGE-M3's real context limit is 8192 tokens (confirmed from its config) - 6000 leaves a safety margin.
MAX_TOKENS = 6000
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")
fallback_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer, chunk_size=MAX_TOKENS, chunk_overlap=200, separators=["\n\n", ". ", " ", ""]
)

# Segments under this size after the citation filter get merged into their neighbor instead of
# standing alone as a near-empty, mislabeled chunk.
MIN_SEGMENT_CHARS = 40


def article_no_from_match(m: re.Match) -> str | None:
    if m.group(1):
        return m.group(1)
    if m.group(2):
        return m.group(2)
    matched_text = m.group(0)
    for word, digit in ORDINAL_TO_DIGIT.items():
        if word in matched_text:
            return digit
    return None


def is_citation(text: str, m: re.Match) -> bool:
    """True if match is a citation reference, not a real article heading."""
    tail = text[m.end():m.end() + 40]
    cm = CITATION_PHRASE.search(tail)
    if not cm:
        return False
    after = tail[cm.end():cm.end() + 15]
    return bool(re.match(r"^[\s,،]*[.،]", after)) or after.strip() == ""


def merge_tiny_segments(segments: list[dict], min_chars: int) -> list[dict]:
    """Merge any segment under min_chars into the segment that follows it."""
    if not segments:
        return segments
    out = [dict(segments[0])]
    for seg in segments[1:]:
        if len(out[-1]["text"]) < min_chars:
            out[-1]["text"] = (out[-1]["text"] + " " + seg["text"]).strip()
        else:
            out.append(dict(seg))
    if len(out) > 1 and len(out[-1]["text"]) < min_chars:
        last = out.pop()
        out[-1]["text"] = (out[-1]["text"] + " " + last["text"]).strip()
    return out


def segment_legislation_by_article(text: str) -> list[dict]:
    """One segment per detected article; citation-only matches excluded, tiny leftovers merged."""
    all_matches = list(ARTICLE_HEADER.finditer(text))
    matches = [m for m in all_matches if not is_citation(text, m)]
    if not matches:
        return [{"text": text, "article_no": None}]
    segments = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        segments.append({"text": text[start:end].strip(), "article_no": article_no_from_match(m)})
    return merge_tiny_segments(segments, MIN_SEGMENT_CHARS)


def n_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))


def load_lloc_documents(path: Path) -> list[Document]:
    records = json.loads(path.read_text(encoding="utf-8"))
    docs = []
    for r in records:
        if not r.get("normalized_text"):
            continue
        base_meta = {
            "source": "lloc",
            "doc_id": r["code"],
            "title": r.get("title") or "",
            "categories": ", ".join(r.get("categories", [])),
        }
        for seg in segment_legislation_by_article(r["normalized_text"]):
            docs.append(Document(page_content=seg["text"], metadata={**base_meta, "article_no": seg["article_no"]}))
    return docs


SJC_NO_JUDGMENT_PLACEHOLDER = "مجموعة الاحكام الصادرة من محكمة التمييز لا يوجد"


def load_judgment_documents(path: Path, source: str, id_field: str) -> list[Document]:
    """Skips sjc.bh's placeholder text for cases with no judgment published."""
    records = json.loads(path.read_text(encoding="utf-8"))
    docs = []
    skipped_placeholder = 0
    for r in records:
        if not r.get("normalized_text"):
            continue
        if r["normalized_text"].strip() == SJC_NO_JUDGMENT_PLACEHOLDER:
            skipped_placeholder += 1
            continue
        doc_id = r.get(id_field) or r.get("key") or r.get("case_id")
        metadata = {"source": source, "doc_id": doc_id}
        if source == "sjc" and doc_id:
            parts = doc_id.split(" ")
            type_code = parts[1] if len(parts) > 1 else None
            metadata["case_type"] = SJC_TYPE_CODES.get(type_code)
        docs.append(Document(page_content=r["normalized_text"], metadata=metadata))
    if skipped_placeholder:
        print(f"{source}: skipped {skipped_placeholder} 'no judgment available' placeholder records")
    return docs


lloc_docs = load_lloc_documents(PROCESSED_DIR / "lloc_normalized.json")
sjc_docs = load_judgment_documents(PROCESSED_DIR / "sjc_normalized.json", "sjc", "key")
ccb_docs = load_judgment_documents(PROCESSED_DIR / "ccb_normalized.json", "ccb", "case_id")

print(f"lloc: {len(lloc_docs)} per-article documents")
print(f"sjc:  {len(sjc_docs)} per-case documents")
print(f"ccb:  {len(ccb_docs)} per-case documents")

missing_type = sum(1 for d in sjc_docs if not d.metadata.get("case_type"))
print(f"sjc documents missing case_type: {missing_type}")

### Legislation — one chunk per article, fallback split only if oversized

In [ ]:
def split_with_fallback(doc: Document) -> list[Document]:
    """One chunk per document by default; subdivided only if it exceeds MAX_TOKENS. Sub-chunks
    keep the parent's full metadata plus a sub_chunk_index."""
    if n_tokens(doc.page_content) <= MAX_TOKENS:
        return [doc]
    pieces = fallback_splitter.split_text(doc.page_content)
    return [
        Document(page_content=piece, metadata={**doc.metadata, "sub_chunk_index": i})
        for i, piece in enumerate(pieces)
    ]


lloc_splits = []
lloc_oversized = 0
for d in lloc_docs:
    pieces = split_with_fallback(d)
    if len(pieces) > 1:
        lloc_oversized += 1
    lloc_splits.extend(pieces)

with_article_no = sum(1 for d in lloc_splits if d.metadata.get("article_no"))
print(f"{len(lloc_docs)} article-segments -> {len(lloc_splits)} chunks "
      f"({lloc_oversized} articles needed a fallback split, {with_article_no} chunks tagged with an article_no)")
lloc_splits[1].page_content[:300]

### Judgments/rulings — one chunk per whole case, fallback split only if oversized

In [9]:
sjc_splits = []
sjc_oversized = 0
for d in sjc_docs:
    pieces = split_with_fallback(d)
    if len(pieces) > 1:
        sjc_oversized += 1
    sjc_splits.extend(pieces)

ccb_splits = []
ccb_oversized = 0
for d in ccb_docs:
    pieces = split_with_fallback(d)
    if len(pieces) > 1:
        ccb_oversized += 1
    ccb_splits.extend(pieces)

print(f"sjc: {len(sjc_docs)} cases -> {len(sjc_splits)} chunks ({sjc_oversized} cases needed a fallback split)")
print(f"ccb: {len(ccb_docs)} cases -> {len(ccb_splits)} chunks ({ccb_oversized} cases needed a fallback split)")

sjc: 9094 cases -> 10193 chunks (536 cases needed a fallback split)
ccb: 83 cases -> 126 chunks (34 cases needed a fallback split)


### Save the split documents for the next stage

In [10]:
import pickle

all_splits = lloc_splits + sjc_splits + ccb_splits
OUT_PATH = Path("../data/processed/document_splits.pkl")
OUT_PATH.write_bytes(pickle.dumps(all_splits))
print(f"Saved {len(all_splits)} total chunks -> {OUT_PATH}")

Saved 26977 total chunks -> ..\data\processed\document_splits.pkl
